# ⭐ Day 14: Pandas Data Cleaning & Missing Values
Critical Preprocessing Step for AI & ML | Step-by-Step Tutorial with Examples & Exercises

**Day 14 of 369-day Python & AI Learning Path** 💡

## Introduction

Welcome to Day 14 of your Python & AI journey! Today we tackle what data scientists spend **60-80% of their time** on: **data cleaning and preprocessing**. While building neural networks and tuning hyperparameters gets the glory, the reality is that real-world AI/ML projects begin and end with messy data that needs meticulous cleaning.

**Why is this skill so critical?** First, real-world datasets are almost never clean. They contain missing values, inconsistent formats, duplicates, outliers, and incorrect data types. Machine learning algorithms—whether traditional statistical models or modern deep learning architectures—are fundamentally mathematical operations that break when fed null values or strings where numbers should be. A single missing value can halt your entire training pipeline.

Second, **how** you handle missing data directly impacts model performance and validity. Simply dropping rows with missing values might eliminate your most important samples, creating selection bias. Poor imputation strategies can distort feature distributions, leading to inaccurate predictions. In medical AI or financial modeling, these errors have serious real-world consequences. Understanding Missing Completely At Random (MCAR), Missing At Random (MAR), and Missing Not At Random (MNAR) patterns helps you choose appropriate strategies rather than applying one-size-fits-all solutions.

Third, data cleaning is where domain expertise shines. Knowing whether a zero represents a true measurement or a missing value, recognizing impossible combinations (like a 5-year-old with a 20-year employment history), and understanding which features can be reasonably imputed versus which require specialized handling—these decisions require both technical skill and business context. Master these techniques today, and you'll save countless hours of debugging downstream model issues. Clean data is the foundation of trustworthy AI! 🧹✨

## 📌 Table of Contents

1. [Detecting Missing Values](#detect)
2. [Understanding Missing Data Mechanisms](#mechanisms)
3. [Dropping Missing Values](#drop)
4. [Filling Missing Values: Simple Methods](#fill-simple)
5. [Forward/Backward Fill](#ffill)
6. [Group-Wise Imputation](#group)
7. [Advanced Imputation with sklearn](#sklearn)
8. [Handling Duplicates](#duplicates)
9. [Data Type Conversion](#dtypes)
10. [String Cleaning](#strings)
11. [Outlier Detection Preview](#outliers)
12. [Building a Cleaning Pipeline](#pipeline)
13. [🛠️ Hands-On Exercises](#exercises)
14. [Solutions](#solutions)
15. [What's Next: Day 15](#next)

## 1. Detecting Missing Values {#detect}

The first step in cleaning is understanding the scope of the problem. Pandas provides several tools to identify and quantify missing data.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(exist_ok=True)
plt.switch_backend('Agg')


# Set random seed for reproducibility
np.random.seed(42)

# Create synthetic medical dataset with realistic missing patterns
n = 1000
data = {
    'patient_id': range(1000, 2000),
    'age': np.random.randint(18, 90, n),
    'gender': np.random.choice(['M', 'F', None], n, p=[0.48, 0.48, 0.04]),
    'blood_pressure': np.random.normal(120, 20, n).round(1),
    'cholesterol': np.random.normal(200, 40, n).round(1),
    'bmi': np.random.normal(25, 5, n).round(2),
    'smoking_status': np.random.choice(['never', 'former', 'current', None], n, p=[0.5, 0.25, 0.2, 0.05]),
    'diabetes': np.random.choice([0, 1, None], n, p=[0.85, 0.1, 0.05]),
    'heart_disease': np.random.choice([0, 1], n, p=[0.9, 0.1]),
    'last_visit_date': pd.date_range('2023-01-01', periods=n, freq='D').to_series().sample(n, replace=True).values
}

df = pd.DataFrame(data)

# Introduce systematic missing values (MAR pattern - missingness related to observed data)
# Older patients more likely to have missing cholesterol (maybe they skip tests)
old_patient_mask = df['age'] > 70
old_patient_idx = df.index[old_patient_mask]
missing_old_idx = old_patient_idx[np.random.random(old_patient_mask.sum()) > 0.6]
df.loc[missing_old_idx, 'cholesterol'] = np.nan

# Higher BMI patients more likely to have missing smoking status (social desirability bias)
high_bmi_mask = df['bmi'] > 30
high_bmi_idx = df.index[high_bmi_mask]
missing_smoking_idx = high_bmi_idx[np.random.random(high_bmi_mask.sum()) > 0.7]
df.loc[missing_smoking_idx, 'smoking_status'] = None

# Random missing values in blood pressure (MCAR)
random_missing = np.random.choice(df.index, size=50, replace=False)
df.loc[random_missing, 'blood_pressure'] = np.nan

print("Dataset created with intentional missing patterns")
print(f"Shape: {df.shape}")
print("\nFirst 5 rows:")
print(df.head())

Dataset created with intentional missing patterns
Shape: (1000, 10)

First 5 rows:
   patient_id  age gender  blood_pressure  cholesterol    bmi smoking_status  \
0        1000   69      F           133.2        199.9  22.65        current   
1        1001   32      M           148.7        250.9  32.25          never   
2        1002   89      F           123.3          NaN  29.76        current   
3        1003   78      M           102.9        204.5  34.15         former   
4        1004   38      M             NaN        184.1  27.65          never   

  diabetes  heart_disease last_visit_date  
0        0              0      2024-07-04  
1        0              0      2025-03-12  
2        0              0      2023-12-19  
3        0              0      2023-10-27  
4        0              0      2024-11-12  


In [2]:
# Detect missing values using multiple methods
print("=== Detecting Missing Values ===\n")

# Method 1: isna() / isnull() - returns boolean DataFrame
print("1. Boolean mask (isna()) for first 5 rows:")
print(df.isna().head())
print()

# Method 2: Sum by column to get counts
print("2. Missing value counts per column:")
missing_counts = df.isnull().sum()
print(missing_counts)
print()

# Method 3: Percentage of missing values
print("3. Missing value percentages:")
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
print(missing_pct)
print()

# Method 4: info() shows non-null counts
print("4. DataFrame info():")
df.info()
print()

# Method 5: Total missing values in dataset
total_missing = df.isnull().sum().sum()
print(f"5. Total missing values in entire dataset: {total_missing}")
print(f"   Percentage of all data points: {(total_missing / (df.shape[0] * df.shape[1]) * 100):.2f}%")

=== Detecting Missing Values ===

1. Boolean mask (isna()) for first 5 rows:
   patient_id    age  gender  blood_pressure  cholesterol    bmi  \
0       False  False   False           False        False  False   
1       False  False   False           False        False  False   
2       False  False   False           False         True  False   
3       False  False   False           False        False  False   
4       False  False   False            True        False  False   

   smoking_status  diabetes  heart_disease  last_visit_date  
0           False     False          False            False  
1           False     False          False            False  
2           False     False          False            False  
3           False     False          False            False  
4           False     False          False            False  

2. Missing value counts per column:
patient_id           0
age                  0
gender              42
blood_pressure      50
cholesterol  

In [3]:
# Visualization: Missing value heatmap
plt.figure(figsize=(12, 6))
sns.heatmap(df.isnull(), cbar=True, yticklabels=False, cmap='viridis', 
            xticklabels=df.columns, cbar_kws={'label': 'Missing Value'})
plt.title('Missing Value Heatmap\n(Yellow = Missing, Dark = Present)', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'missing_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved missing value heatmap")

# Bar plot of missing counts
plt.figure(figsize=(10, 6))
missing_counts = df.isnull().sum()
missing_counts = missing_counts[missing_counts > 0]  # Only show columns with missing values
colors = plt.cm.Set3(np.linspace(0, 1, len(missing_counts)))
bars = plt.bar(missing_counts.index, missing_counts.values, color=colors, edgecolor='black')
plt.title('Missing Value Counts by Column', fontsize=14, fontweight='bold')
plt.xlabel('Columns')
plt.ylabel('Number of Missing Values')
plt.xticks(rotation=45, ha='right')
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{int(height)}', ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'missing_barplot.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved missing value bar plot")

C:\Users\786\AppData\Local\Temp\ipykernel_2180\2000902142.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saved missing value heatmap
Saved missing value bar plot


C:\Users\786\AppData\Local\Temp\ipykernel_2180\2000902142.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Understanding Missing Data Mechanisms {#mechanisms}

**Why** data is missing determines **how** you should handle it. Statisticians classify missingness into three categories:

- **MCAR (Missing Completely At Random)**: Missingness has no relationship to any data. Like random technical errors. Safest to drop or simple impute.
- **MAR (Missing At Random)**: Missingness relates to *observed* data but not the missing value itself. Example: older patients skip cholesterol tests. Requires careful handling—dropping creates bias toward younger patients.
- **MNAR (Missing Not At Random)**: Missingness relates to the *missing value itself*. Example: people with high income refuse to report it. Most dangerous—requires domain expertise and specialized models.

In our dataset:
- `blood_pressure`: MCAR (random 50 missing)
- `cholesterol`: MAR (related to age)
- `smoking_status`: MNAR-like (related to BMI, which correlates with smoking)

In [4]:
# Analyze missing patterns
print("=== Analyzing Missing Data Patterns ===\n")

# Check if cholesterol missingness relates to age (MAR detection)
cholesterol_missing = df['cholesterol'].isnull()
print("Cholesterol missing by age group:")
print(df.groupby(cholesterol_missing)['age'].agg(['mean', 'count']))
print()

# Check if smoking missing relates to BMI
smoking_missing = df['smoking_status'].isnull()
print("Smoking status missing by BMI group:")
print(df.groupby(smoking_missing)['bmi'].agg(['mean', 'count']))
print()

# Visualize MAR pattern: Age vs Cholesterol missing
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Age distribution by cholesterol availability
df[df['cholesterol'].notnull()]['age'].hist(bins=30, alpha=0.7, label='Cholesterol Present', ax=axes[0], color='skyblue')
df[df['cholesterol'].isnull()]['age'].hist(bins=30, alpha=0.7, label='Cholesterol Missing', ax=axes[0], color='salmon')
axes[0].set_title('Age Distribution by Cholesterol Availability\n(MAR Pattern)', fontweight='bold')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Count')
axes[0].legend()

# BMI distribution by smoking availability
df[df['smoking_status'].notnull()]['bmi'].hist(bins=30, alpha=0.7, label='Smoking Known', ax=axes[1], color='lightgreen')
df[df['smoking_status'].isnull()]['bmi'].hist(bins=30, alpha=0.7, label='Smoking Missing', ax=axes[1], color='orange')
axes[1].set_title('BMI Distribution by Smoking Status Availability\n(MNAR-like Pattern)', fontweight='bold')
axes[1].set_xlabel('BMI')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'missing_patterns.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved missing pattern analysis plots")

=== Analyzing Missing Data Patterns ===

Cholesterol missing by age group:
                  mean  count
cholesterol                  
False        49.797327    898
True         80.029412    102

Smoking status missing by BMI group:
                     mean  count
smoking_status                  
False           24.587891    915
True            28.098471     85

Saved missing pattern analysis plots


C:\Users\786\AppData\Local\Temp\ipykernel_2180\3159552554.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Dropping Missing Values {#drop}

`dropna()` removes missing values but should be used cautiously. Dropping too much data reduces statistical power and can introduce bias if missingness isn't MCAR.

In [5]:
# Dropping missing values
print("=== Dropping Missing Values ===\n")

# Original shape
print(f"Original shape: {df.shape}")
print(f"Original missing values: {df.isnull().sum().sum()}\n")

# Option 1: Drop rows with ANY missing values (dangerous!)
df_drop_any = df.dropna()
print(f"After dropna() (any missing): {df_drop_any.shape}")
print(f"Rows removed: {df.shape[0] - df_drop_any.shape[0]} ({(1 - df_drop_any.shape[0]/df.shape[0])*100:.1f}%)")
print()

# Option 2: Drop rows where ALL values are missing (not applicable here)
df_drop_all = df.dropna(how='all')
print(f"After dropna(how='all'): {df_drop_all.shape} (same as original if no empty rows)")
print()

# Option 3: Drop rows with missing values in specific columns only
df_drop_specific = df.dropna(subset=['blood_pressure', 'age'])
print(f"After dropna on blood_pressure and age: {df_drop_specific.shape}")
print()

# Option 4: Drop columns with more than 20% missing
threshold = len(df) * 0.8  # Keep columns with at least 80% non-null
df_drop_cols = df.dropna(axis=1, thresh=threshold)
print(f"After dropping columns with >20% missing: {df_drop_cols.shape}")
print(f"Remaining columns: {df_drop_cols.columns.tolist()}")
print()

# Option 5: Drop duplicates (preview)
print(f"Duplicate rows: {df.duplicated().sum()}")

=== Dropping Missing Values ===

Original shape: (1000, 10)
Original missing values: 326

After dropna() (any missing): (720, 10)
Rows removed: 280 (28.0%)

After dropna(how='all'): (1000, 10) (same as original if no empty rows)

After dropna on blood_pressure and age: (950, 10)

After dropping columns with >20% missing: (1000, 10)
Remaining columns: ['patient_id', 'age', 'gender', 'blood_pressure', 'cholesterol', 'bmi', 'smoking_status', 'diabetes', 'heart_disease', 'last_visit_date']

Duplicate rows: 0


## 4. Filling Missing Values: Simple Methods {#fill-simple}

Imputation replaces missing values with statistical estimates. Choose methods based on data distribution and missing mechanism.

In [6]:
# Simple imputation methods
print("=== Simple Imputation Methods ===\n")

# Make a copy for imputation experiments
df_impute = df.copy()

# Method 1: Fill with constant value
df_impute['gender'] = df_impute['gender'].fillna('Unknown')
print("1. Filled missing gender with 'Unknown'")
print(f"   Gender value counts:\n{df_impute['gender'].value_counts()}\n")

# Method 2: Fill with mean (appropriate for normally distributed data)
bp_mean = df_impute['blood_pressure'].mean()
df_impute['blood_pressure'] = df_impute['blood_pressure'].fillna(bp_mean)
print(f"2. Filled missing blood_pressure with mean: {bp_mean:.1f}")
print(f"   Missing after: {df_impute['blood_pressure'].isnull().sum()}\n")

# Method 3: Fill with median (robust to outliers)
chol_median = df_impute['cholesterol'].median()
df_impute['cholesterol_mean'] = df['cholesterol'].fillna(df['cholesterol'].mean())
df_impute['cholesterol_median'] = df['cholesterol'].fillna(chol_median)
print(f"3. Filled cholesterol - Mean: {df['cholesterol'].mean():.1f}, Median: {chol_median:.1f}")
print(f"   Std dev original: {df['cholesterol'].std():.1f}")
print(f"   Std dev after mean imputation: {df_impute['cholesterol_mean'].std():.1f}")
print(f"   Std dev after median imputation: {df_impute['cholesterol_median'].std():.1f}\n")

# Method 4: Fill with mode (for categorical data)
smoking_mode = df['smoking_status'].mode()[0] if not df['smoking_status'].mode().empty else 'never'
df_impute['smoking_status'] = df_impute['smoking_status'].fillna(smoking_mode)
print(f"4. Filled missing smoking_status with mode: '{smoking_mode}'")
print(f"   Missing after: {df_impute['smoking_status'].isnull().sum()}\n")

# Method 5: Fill with specific value per column using dictionary
df_filled = df.fillna({
    'gender': 'Unknown',
    'diabetes': 0,  # Assume not diabetic if unknown
    'blood_pressure': df['blood_pressure'].median()
})
print("5. Dictionary-based filling applied")
print(f"   Remaining missing: {df_filled.isnull().sum().sum()}")

=== Simple Imputation Methods ===

1. Filled missing gender with 'Unknown'
   Gender value counts:
gender
F          486
M          472
Unknown     42
Name: count, dtype: int64

2. Filled missing blood_pressure with mean: 120.4
   Missing after: 0

3. Filled cholesterol - Mean: 200.7, Median: 200.4
   Std dev original: 39.4
   Std dev after mean imputation: 37.4
   Std dev after median imputation: 37.4

4. Filled missing smoking_status with mode: 'never'
   Missing after: 0

5. Dictionary-based filling applied
   Remaining missing: 187


In [7]:
# Visualization: Distribution before and after imputation
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Cholesterol - Original vs Mean Imputed
axes[0, 0].hist(df['cholesterol'].dropna(), bins=30, alpha=0.7, color='skyblue', edgecolor='black', label='Original')
axes[0, 0].axvline(df['cholesterol'].mean(), color='red', linestyle='--', label=f'Mean: {df["cholesterol"].mean():.1f}')
axes[0, 0].set_title('Cholesterol: Original Distribution', fontweight='bold')
axes[0, 0].set_xlabel('Cholesterol Level')
axes[0, 0].set_ylabel('Count')
axes[0, 0].legend()

axes[0, 1].hist(df_impute['cholesterol_mean'], bins=30, alpha=0.7, color='lightcoral', edgecolor='black', label='Mean Imputed')
axes[0, 1].axvline(df_impute['cholesterol_mean'].mean(), color='red', linestyle='--', label=f'Mean: {df_impute["cholesterol_mean"].mean():.1f}')
axes[0, 1].set_title('Cholesterol: After Mean Imputation\n(Note the spike at mean value)', fontweight='bold')
axes[0, 1].set_xlabel('Cholesterol Level')
axes[0, 1].set_ylabel('Count')
axes[0, 1].legend()

# Blood Pressure - Original vs Imputed
axes[1, 0].hist(df['blood_pressure'].dropna(), bins=30, alpha=0.7, color='lightgreen', edgecolor='black')
axes[1, 0].axvline(df['blood_pressure'].mean(), color='red', linestyle='--')
axes[1, 0].set_title('Blood Pressure: Original', fontweight='bold')
axes[1, 0].set_xlabel('Blood Pressure')
axes[1, 0].set_ylabel('Count')

axes[1, 1].hist(df_impute['blood_pressure'], bins=30, alpha=0.7, color='gold', edgecolor='black')
axes[1, 1].axvline(df_impute['blood_pressure'].mean(), color='red', linestyle='--')
axes[1, 1].set_title('Blood Pressure: After Mean Imputation', fontweight='bold')
axes[1, 1].set_xlabel('Blood Pressure')
axes[1, 1].set_ylabel('Count')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'imputation_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved imputation comparison plots")

Saved imputation comparison plots


C:\Users\786\AppData\Local\Temp\ipykernel_2180\3819524606.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Forward/Backward Fill {#ffill}

Time-series data often uses forward fill (`ffill`) or backward fill (`bfill`) to propagate last known values. This assumes data doesn't change rapidly between observations.

In [8]:
# Forward and backward fill (useful for time series)
print("=== Forward/Backward Fill ===\n")

# Create time series data with gaps
dates = pd.date_range('2024-01-01', periods=10, freq='D')
ts_data = pd.DataFrame({
    'date': dates,
    'temperature': [22, 23, np.nan, np.nan, 25, 24, np.nan, 26, np.nan, 28],
    'humidity': [60, np.nan, 58, 57, np.nan, np.nan, 55, 54, np.nan, 50]
})
ts_data = ts_data.set_index('date')

print("Original time series:")
print(ts_data)
print()

# Forward fill: propagate last valid observation forward
ts_ffill = ts_data.ffill()
print("After forward fill (ffill):")
print(ts_ffill)
print()

# Backward fill: propagate next valid observation backward
ts_bfill = ts_data.bfill()
print("After backward fill (bfill):")
print(ts_bfill)
print()

# Limit the fill (only fill up to 1 consecutive missing)
ts_limited = ts_data.ffill(limit=1)
print("After ffill with limit=1:")
print(ts_limited)
print()

# Interpolate (linear interpolation between points)
ts_interpolated = ts_data.interpolate(method='linear')
print("After linear interpolation:")
print(ts_interpolated)

=== Forward/Backward Fill ===

Original time series:
            temperature  humidity
date                             
2024-01-01         22.0      60.0
2024-01-02         23.0       NaN
2024-01-03          NaN      58.0
2024-01-04          NaN      57.0
2024-01-05         25.0       NaN
2024-01-06         24.0       NaN
2024-01-07          NaN      55.0
2024-01-08         26.0      54.0
2024-01-09          NaN       NaN
2024-01-10         28.0      50.0

After forward fill (ffill):
            temperature  humidity
date                             
2024-01-01         22.0      60.0
2024-01-02         23.0      60.0
2024-01-03         23.0      58.0
2024-01-04         23.0      57.0
2024-01-05         25.0      57.0
2024-01-06         24.0      57.0
2024-01-07         24.0      55.0
2024-01-08         26.0      54.0
2024-01-09         26.0      54.0
2024-01-10         28.0      50.0

After backward fill (bfill):
            temperature  humidity
date                             
2024

## 6. Group-Wise Imputation {#group}

More sophisticated than simple imputation: fill missing values based on group statistics (e.g., mean cholesterol by age group rather than overall mean). This preserves group differences and often performs better for MAR data.

In [9]:
# Group-wise imputation
print("=== Group-Wise Imputation ===\n")

# Create age groups for stratified imputation
df_group = df.copy()
df_group['age_group'] = pd.cut(df_group['age'], bins=[0, 40, 60, 100], labels=['Young', 'Middle', 'Senior'])

print("Cholesterol missing by age group:")
print(df_group.groupby('age_group')['cholesterol'].apply(lambda x: x.isnull().sum()))
print()

# Calculate mean cholesterol by age group
group_means = df_group.groupby('age_group')['cholesterol'].transform('mean')
print("Mean cholesterol by age group:")
print(df_group.groupby('age_group')['cholesterol'].mean())
print()

# Fill missing cholesterol with group mean
df_group['cholesterol_imputed'] = df_group['cholesterol'].fillna(group_means)
print("After group-wise imputation:")
print(f"Missing values: {df_group['cholesterol_imputed'].isnull().sum()}")
print()

# Compare overall mean vs group mean imputation
overall_mean_fill = df['cholesterol'].fillna(df['cholesterol'].mean())
group_mean_fill = df_group['cholesterol_imputed']

print("Standard deviation comparison:")
print(f"Original: {df['cholesterol'].std():.2f}")
print(f"Overall mean imputation: {overall_mean_fill.std():.2f}")
print(f"Group mean imputation: {group_mean_fill.std():.2f}")
print("\nGroup-wise imputation preserves variance better!")

=== Group-Wise Imputation ===

Cholesterol missing by age group:
age_group
Young       0
Middle      0
Senior    102
Name: cholesterol, dtype: int64

Mean cholesterol by age group:
age_group
Young     201.330675
Middle    199.525275
Senior    201.170569
Name: cholesterol, dtype: float64

After group-wise imputation:
Missing values: 0

Standard deviation comparison:
Original: 39.43
Overall mean imputation: 37.37
Group mean imputation: 37.37

Group-wise imputation preserves variance better!


## 7. Advanced Imputation with sklearn {#sklearn}

For production ML pipelines, scikit-learn's `SimpleImputer` provides a robust, reusable approach that integrates with preprocessing pipelines.

In [10]:
# Advanced imputation with sklearn
print("=== sklearn SimpleImputer ===\n")

from sklearn.impute import SimpleImputer

# Prepare data (numerical columns only for this example)
num_cols = ['age', 'blood_pressure', 'cholesterol', 'bmi']
df_sklearn = df[num_cols + ['heart_disease']].copy()

print("Before imputation:")
print(df_sklearn.isnull().sum())
print()

# Strategy 1: Mean imputation
imputer_mean = SimpleImputer(strategy='mean')
df_mean_imputed = pd.DataFrame(
    imputer_mean.fit_transform(df_sklearn),
    columns=df_sklearn.columns,
    index=df_sklearn.index
)
print("After mean imputation:")
print(df_mean_imputed.isnull().sum().sum())
print()

# Strategy 2: Most frequent (for mixed data)
imputer_median = SimpleImputer(strategy='median')
df_median_imputed = pd.DataFrame(
    imputer_median.fit_transform(df_sklearn),
    columns=df_sklearn.columns,
    index=df_sklearn.index
)
print("Median imputation also completed successfully")
print()

# Strategy 3: Constant value
imputer_constant = SimpleImputer(strategy='constant', fill_value=-999)
df_constant = pd.DataFrame(
    imputer_constant.fit_transform(df_sklearn),
    columns=df_sklearn.columns,
    index=df_sklearn.index
)
print("Constant (-999) imputation completed (useful for tree-based models)")
print()

# The imputer can be saved and reused for test data!
print("Imputer statistics:")
print(f"Mean values stored: {imputer_mean.statistics_}")

=== sklearn SimpleImputer ===

Before imputation:
age                 0
blood_pressure     50
cholesterol       102
bmi                 0
heart_disease       0
dtype: int64

After mean imputation:
0

Median imputation also completed successfully

Constant (-999) imputation completed (useful for tree-based models)

Imputer statistics:
Mean values stored: [5.28810000e+01 1.20355789e+02 2.00728508e+02 2.48862900e+01
 1.08000000e-01]


## 8. Handling Duplicates {#duplicates}

Duplicate records inflate dataset size, bias statistics, and can cause data leakage between train/test sets. Always check for and handle duplicates early.

In [11]:
# Handling duplicates
print("=== Handling Duplicates ===\n")

# Create a dataframe with intentional duplicates
df_dups = pd.concat([df.iloc[:5], df.iloc[:3], df.iloc[10:15]], ignore_index=True)
print(f"Dataset with duplicates shape: {df_dups.shape}")
print(f"Actual unique rows should be: 5 + 3 + 5 = 13, but we have: {len(df_dups)}")
print()

# Detect duplicates
print("1. Check for any duplicates:")
print(f"   Total duplicates: {df_dups.duplicated().sum()}")
print()

# Show which rows are duplicates (keep=False marks all duplicates)
print("2. All duplicate rows (keep=False):")
print(df_dups[df_dups.duplicated(keep=False)])
print()

# Check duplicates based on specific columns
print("3. Duplicates based on age and gender only:")
print(f"   Count: {df_dups.duplicated(subset=['age', 'gender']).sum()}")
print()

# Remove duplicates
print("4. Removing duplicates:")
df_no_dups = df_dups.drop_duplicates()
print(f"   After drop_duplicates(): {df_no_dups.shape}")
print()

# Keep last occurrence instead of first
df_keep_last = df_dups.drop_duplicates(keep='last')
print(f"   After keeping last: {df_keep_last.shape}")
print()

# Remove duplicates based on subset
df_subset_unique = df_dups.drop_duplicates(subset=['age', 'gender'])
print(f"   Unique by age+gender: {df_subset_unique.shape}")

=== Handling Duplicates ===

Dataset with duplicates shape: (13, 10)
Actual unique rows should be: 5 + 3 + 5 = 13, but we have: 13

1. Check for any duplicates:
   Total duplicates: 3

2. All duplicate rows (keep=False):
   patient_id  age gender  blood_pressure  cholesterol    bmi smoking_status  \
0        1000   69      F           133.2        199.9  22.65        current   
1        1001   32      M           148.7        250.9  32.25          never   
2        1002   89      F           123.3          NaN  29.76        current   
5        1000   69      F           133.2        199.9  22.65        current   
6        1001   32      M           148.7        250.9  32.25          never   
7        1002   89      F           123.3          NaN  29.76        current   

  diabetes  heart_disease last_visit_date  
0        0              0      2024-07-04  
1        0              0      2025-03-12  
2        0              0      2023-12-19  
5        0              0      2024-07-04 

## 9. Data Type Conversion {#dtypes}

Incorrect data types waste memory and cause analysis errors. Common issues: numbers stored as strings, dates as objects, categorical data not optimized.

In [12]:
# Data type conversion
print("=== Data Type Conversion ===\n")

# Check current types
print("1. Current data types:")
print(df.dtypes)
print()

# Create messy data for demonstration
df_types = pd.DataFrame({
    'id': ['001', '002', '003'],  # String that should be int
    'price': ['$10.50', '$20.00', '$15.75'],  # String with currency
    'quantity': ['5', '10', '3'],  # String that should be int
    'date': ['2024-01-15', '2024-02-20', '2024-03-10'],  # String that should be datetime
    'percentage': ['95%', '87%', '92%']  # String with %
})

print("2. Messy data types:")
print(df_types.dtypes)
print()

# Convert types
df_types['id'] = df_types['id'].astype(int)
df_types['price'] = df_types['price'].str.replace('$', '').astype(float)
df_types['quantity'] = df_types['quantity'].astype(int)
df_types['date'] = pd.to_datetime(df_types['date'])
df_types['percentage'] = df_types['percentage'].str.replace('%', '').astype(float) / 100

print("3. After conversion:")
print(df_types.dtypes)
print()
print(df_types)
print()

# Memory optimization: convert to categorical
df_memory = df.copy()
print("4. Memory usage before optimization:")
print(f"   Gender as object: {df_memory['gender'].memory_usage(deep=True)} bytes")
df_memory['gender'] = df_memory['gender'].astype('category')
print(f"   Gender as category: {df_memory['gender'].memory_usage(deep=True)} bytes")
print(f"   Memory savings: {(1 - df_memory['gender'].memory_usage(deep=True)/df['gender'].memory_usage(deep=True))*100:.1f}%")
print()

# pd.to_numeric with errors='coerce' (invalid parsing becomes NaN)
messy_numbers = pd.Series(['1', '2', 'three', '4', 'N/A'])
converted = pd.to_numeric(messy_numbers, errors='coerce')
print("5. pd.to_numeric with errors='coerce':")
print(f"   Original: {messy_numbers.tolist()}")
print(f"   Converted: {converted.tolist()}")

=== Data Type Conversion ===

1. Current data types:
patient_id                  int64
age                         int32
gender                        str
blood_pressure            float64
cholesterol               float64
bmi                       float64
smoking_status                str
diabetes                   object
heart_disease               int64
last_visit_date    datetime64[us]
dtype: object

2. Messy data types:
id            str
price         str
quantity      str
date          str
percentage    str
dtype: object

3. After conversion:
id                     int64
price                float64
quantity               int64
date          datetime64[us]
percentage           float64
dtype: object

   id  price  quantity       date  percentage
0   1  10.50         5 2024-01-15        0.95
1   2  20.00        10 2024-02-20        0.87
2   3  15.75         3 2024-03-10        0.92

4. Memory usage before optimization:
   Gender as object: 9215 bytes
   Gender as category: 1151 byt

## 10. String Cleaning {#strings}

Text data is notoriously messy: inconsistent casing, extra whitespace, typos, and mixed formats. Pandas string methods (`.str` accessor) clean these efficiently.

In [13]:
# String cleaning
print("=== String Cleaning ===\n")

# Create messy string data
df_strings = pd.DataFrame({
    'name': ['  John Doe  ', 'JANE SMITH', 'bob johnson', '  Alice  '],
    'email': ['john@email.com', 'JANE@EMAIL.COM', 'bob@test.com', 'alice@domain.COM'],
    'phone': ['(555) 123-4567', '555.987.6543', '555-000-1111', '555 222 3333'],
    'status': ['active', 'ACTIVE', 'Active', 'pending']
})

print("1. Original messy data:")
print(df_strings)
print()

# Clean whitespace
df_strings['name_clean'] = df_strings['name'].str.strip()
print("2. After str.strip() on names:")
print(df_strings[['name', 'name_clean']])
print()

# Standardize case
df_strings['email_lower'] = df_strings['email'].str.lower()
df_strings['name_title'] = df_strings['name_clean'].str.title()
print("3. Standardized case:")
print(df_strings[['email', 'email_lower', 'name_clean', 'name_title']])
print()

# Replace patterns (standardize phone numbers)
df_strings['phone_clean'] = df_strings['phone'].str.replace(r'[\(\)\.\s-]', '', regex=True)
print("4. Standardized phone numbers:")
print(df_strings[['phone', 'phone_clean']])
print()

# Check for substrings
has_gmail = df_strings['email'].str.contains('gmail', case=False, na=False)
print("5. Contains 'gmail':")
print(has_gmail)
print()

# Replace specific values
df_strings['status_standard'] = df_strings['status'].str.lower()
print("6. Standardized status:")
print(df_strings[['status', 'status_standard']].value_counts())

=== String Cleaning ===

1. Original messy data:
           name             email           phone   status
0    John Doe      john@email.com  (555) 123-4567   active
1    JANE SMITH    JANE@EMAIL.COM    555.987.6543   ACTIVE
2   bob johnson      bob@test.com    555-000-1111   Active
3       Alice    alice@domain.COM    555 222 3333  pending

2. After str.strip() on names:
           name   name_clean
0    John Doe       John Doe
1    JANE SMITH   JANE SMITH
2   bob johnson  bob johnson
3       Alice          Alice

3. Standardized case:
              email       email_lower   name_clean   name_title
0    john@email.com    john@email.com     John Doe     John Doe
1    JANE@EMAIL.COM    jane@email.com   JANE SMITH   Jane Smith
2      bob@test.com      bob@test.com  bob johnson  Bob Johnson
3  alice@domain.COM  alice@domain.com        Alice        Alice

4. Standardized phone numbers:
            phone phone_clean
0  (555) 123-4567  5551234567
1    555.987.6543  5559876543
2    555-000-1

## 11. Outlier Detection Preview {#outliers}

Outliers can indicate data entry errors or genuine extreme values. The IQR (Interquartile Range) method is a robust statistical approach for flagging potential outliers.

In [14]:
# Outlier detection using IQR method
print("=== Outlier Detection (IQR Method) ===\n")

def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = data[(data[column] < lower_bound) | (data[column] > upper_bound)]
    return outliers, lower_bound, upper_bound

# Detect outliers in cholesterol
chol_outliers, chol_lower, chol_upper = detect_outliers_iqr(df, 'cholesterol')
print(f"Cholesterol outliers: {len(chol_outliers)} ({len(chol_outliers)/len(df)*100:.1f}%)")
print(f"Valid range: {chol_lower:.1f} to {chol_upper:.1f}")
print(f"Outlier values range: {chol_outliers['cholesterol'].min():.1f} to {chol_outliers['cholesterol'].max():.1f}")
print()

# Detect outliers in BMI
bmi_outliers, bmi_lower, bmi_upper = detect_outliers_iqr(df, 'bmi')
print(f"BMI outliers: {len(bmi_outliers)} ({len(bmi_outliers)/len(df)*100:.1f}%)")
print(f"Valid range: {bmi_lower:.1f} to {bmi_upper:.1f}")
print()

# Visualization: Boxplots for outlier detection
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Cholesterol boxplot
bp_data = [df['cholesterol'].dropna(), chol_outliers['cholesterol']]
axes[0].boxplot(df['cholesterol'].dropna(), vert=True)
axes[0].set_title('Cholesterol Distribution\n(Boxplot shows outliers)', fontweight='bold')
axes[0].set_ylabel('Cholesterol Level')
axes[0].axhline(chol_upper, color='red', linestyle='--', alpha=0.7, label=f'Upper bound: {chol_upper:.1f}')
axes[0].axhline(chol_lower, color='red', linestyle='--', alpha=0.7, label=f'Lower bound: {chol_lower:.1f}')
axes[0].legend()

# BMI boxplot
axes[1].boxplot(df['bmi'].dropna(), vert=True)
axes[1].set_title('BMI Distribution\n(Boxplot shows outliers)', fontweight='bold')
axes[1].set_ylabel('BMI')
axes[1].axhline(bmi_upper, color='red', linestyle='--', alpha=0.7, label=f'Upper bound: {bmi_upper:.1f}')
axes[1].axhline(bmi_lower, color='red', linestyle='--', alpha=0.7, label=f'Lower bound: {bmi_lower:.1f}')
axes[1].legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'outlier_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved outlier detection boxplots")

# Flag outliers without removing (conservative approach)
df['cholesterol_outlier'] = (df['cholesterol'] < chol_lower) | (df['cholesterol'] > chol_upper)
print(f"\nFlagged {df['cholesterol_outlier'].sum()} cholesterol outliers for review")

=== Outlier Detection (IQR Method) ===

Cholesterol outliers: 3 (0.3%)
Valid range: 93.9 to 308.6
Outlier values range: 66.4 to 89.6

BMI outliers: 6 (0.6%)
Valid range: 10.7 to 39.0

Saved outlier detection boxplots

Flagged 3 cholesterol outliers for review


C:\Users\786\AppData\Local\Temp\ipykernel_2180\5063615.py:48: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 12. Building a Cleaning Pipeline {#pipeline}

For production ML, you need reusable, documented cleaning functions. Here's a comprehensive pipeline that combines all techniques.

In [15]:
# Building a comprehensive cleaning pipeline
print("=== Building a Cleaning Pipeline ===\n")

def clean_medical_data(df_raw):
    """
    Comprehensive cleaning pipeline for medical dataset.
    Returns cleaned DataFrame and cleaning report.
    """
    df_clean = df_raw.copy()
    report = {
        'original_shape': df_clean.shape,
        'steps': []
    }
    
    # Step 1: Remove exact duplicates
    initial_rows = len(df_clean)
    df_clean = df_clean.drop_duplicates()
    duplicates_removed = initial_rows - len(df_clean)
    report['steps'].append(f"Removed {duplicates_removed} duplicate rows")
    
    # Step 2: Handle missing critical identifiers
    df_clean = df_clean.dropna(subset=['patient_id', 'age'])
    report['steps'].append(f"Dropped rows with missing patient_id or age: {len(df_clean)} rows remaining")
    
    # Step 3: Impute numerical values with group-wise strategy
    # Blood pressure: median by age group
    age_groups = pd.cut(df_clean['age'], bins=[0, 40, 60, 100], labels=['young', 'middle', 'senior'])
    df_clean['blood_pressure'] = df_clean['blood_pressure'].fillna(
        df_clean.groupby(age_groups)['blood_pressure'].transform('median')
    )
    report['steps'].append("Imputed blood_pressure using median by age group")
    
    # Cholesterol: mean by age group
    df_clean['cholesterol'] = df_clean['cholesterol'].fillna(
        df_clean.groupby(age_groups)['cholesterol'].transform('mean')
    )
    report['steps'].append("Imputed cholesterol using mean by age group")
    
    # BMI: overall median (normally distributed)
    df_clean['bmi'] = df_clean['bmi'].fillna(df_clean['bmi'].median())
    report['steps'].append("Imputed BMI using overall median")
    
    # Step 4: Impute categorical values
    df_clean['gender'] = df_clean['gender'].fillna('Unknown')
    df_clean['smoking_status'] = df_clean['smoking_status'].fillna('unknown')
    df_clean['diabetes'] = df_clean['diabetes'].fillna(0)  # Assume negative if unknown
    report['steps'].append("Imputed categorical variables with 'Unknown' or 0")
    
    # Step 5: Data type optimization
    df_clean['gender'] = df_clean['gender'].astype('category')
    df_clean['smoking_status'] = df_clean['smoking_status'].astype('category')
    df_clean['diabetes'] = df_clean['diabetes'].astype(int)
    df_clean['heart_disease'] = df_clean['heart_disease'].astype(int)
    report['steps'].append("Optimized data types (categorical, int)")
    
    # Step 6: Flag outliers (don't remove, just flag)
    for col in ['blood_pressure', 'cholesterol', 'bmi']:
        Q1 = df_clean[col].quantile(0.25)
        Q3 = df_clean[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        df_clean[f'{col}_outlier'] = (df_clean[col] < lower) | (df_clean[col] > upper)
    report['steps'].append("Flagged outliers in vital measurements")
    
    # Step 7: Create derived features
    df_clean['bmi_category'] = pd.cut(df_clean['bmi'], 
                                       bins=[0, 18.5, 25, 30, 100],
                                       labels=['underweight', 'normal', 'overweight', 'obese'])
    report['steps'].append("Created BMI categories")
    
    report['final_shape'] = df_clean.shape
    report['final_missing'] = df_clean.isnull().sum().sum()
    
    return df_clean, report

# Apply the pipeline
df_cleaned, cleaning_report = clean_medical_data(df)

print("Cleaning Report:")
print(f"Original: {cleaning_report['original_shape']}")
print(f"Final: {cleaning_report['final_shape']}")
print(f"Remaining missing values: {cleaning_report['final_missing']}")
print("\nSteps applied:")
for i, step in enumerate(cleaning_report['steps'], 1):
    print(f"  {i}. {step}")

print("\nCleaned data sample:")
print(df_cleaned.head())

=== Building a Cleaning Pipeline ===

Cleaning Report:
Original: (1000, 11)
Final: (1000, 14)
Remaining missing values: 0

Steps applied:
  1. Removed 0 duplicate rows
  2. Dropped rows with missing patient_id or age: 1000 rows remaining
  3. Imputed blood_pressure using median by age group
  4. Imputed cholesterol using mean by age group
  5. Imputed BMI using overall median
  6. Imputed categorical variables with 'Unknown' or 0
  7. Optimized data types (categorical, int)
  8. Flagged outliers in vital measurements
  9. Created BMI categories

Cleaned data sample:
   patient_id  age gender  blood_pressure  cholesterol    bmi smoking_status  \
0        1000   69      F           133.2   199.900000  22.65        current   
1        1001   32      M           148.7   250.900000  32.25          never   
2        1002   89      F           123.3   201.170569  29.76        current   
3        1003   78      M           102.9   204.500000  34.15         former   
4        1004   38      M  

In [16]:
# Final visualization: Before vs After cleaning comparison
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Original missing patterns
missing_before = df.isnull().sum()
missing_before = missing_before[missing_before > 0]
axes[0, 0].bar(missing_before.index, missing_before.values, color='salmon', edgecolor='black')
axes[0, 0].set_title('Before: Missing Values by Column', fontweight='bold')
axes[0, 0].set_ylabel('Count')
axes[0, 0].tick_params(axis='x', rotation=45)

# After cleaning (should be minimal)
missing_after = df_cleaned[['blood_pressure', 'cholesterol', 'bmi', 'gender', 'smoking_status', 'diabetes']].isnull().sum()
axes[1, 0].bar(missing_after.index, missing_after.values, color='lightgreen', edgecolor='black')
axes[1, 0].set_title('After: Missing Values by Column', fontweight='bold')
axes[1, 0].set_ylabel('Count')
axes[1, 0].tick_params(axis='x', rotation=45)

# Cholesterol distribution comparison
axes[0, 1].hist(df['cholesterol'].dropna(), bins=30, alpha=0.7, color='skyblue', edgecolor='black')
axes[0, 1].set_title('Before: Cholesterol Distribution', fontweight='bold')
axes[0, 1].set_xlabel('Cholesterol')

axes[1, 1].hist(df_cleaned['cholesterol'], bins=30, alpha=0.7, color='lightcoral', edgecolor='black')
axes[1, 1].set_title('After: Cholesterol (Group-Imputed)', fontweight='bold')
axes[1, 1].set_xlabel('Cholesterol')

# Data completeness overview
completeness_before = (1 - df.isnull().sum() / len(df)) * 100
completeness_after = (1 - df_cleaned.isnull().sum() / len(df_cleaned)) * 100
cols = ['blood_pressure', 'cholesterol', 'bmi']
x = np.arange(len(cols))
width = 0.35
axes[0, 2].bar(x - width/2, [completeness_before[col] for col in cols], width, label='Before', color='skyblue', edgecolor='black')
axes[0, 2].bar(x + width/2, [completeness_after[col] for col in cols], width, label='After', color='lightgreen', edgecolor='black')
axes[0, 2].set_title('Data Completeness by Column (%)', fontweight='bold')
axes[0, 2].set_ylabel('Completeness %')
axes[0, 2].set_xticks(x)
axes[0, 2].set_xticklabels(cols)
axes[0, 2].legend()
axes[0, 2].set_ylim(0, 110)

# BMI categories after cleaning
bmi_counts = df_cleaned['bmi_category'].value_counts()
axes[1, 2].bar(bmi_counts.index, bmi_counts.values, color=['lightblue', 'lightgreen', 'gold', 'salmon'], edgecolor='black')
axes[1, 2].set_title('BMI Categories (Derived Feature)', fontweight='bold')
axes[1, 2].set_ylabel('Count')
axes[1, 2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'cleaning_pipeline_results.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved cleaning pipeline comparison")

Saved cleaning pipeline comparison


C:\Users\786\AppData\Local\Temp\ipykernel_2180\1125065312.py:52: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 🛠️ Hands-On Exercises {#exercises}

Practice these data cleaning skills with the medical dataset. Complete each exercise before checking solutions!

### Exercise 1: Detect and Summarize Missing Values
Write code to print a summary showing: (a) total missing values per column, (b) percentage missing per column, and (c) the column with the most missing values.

In [17]:
# your code here


### Exercise 2: Apply Different Imputation Strategies
Create three versions of the `cholesterol` column: (1) filled with overall mean, (2) filled with median by `gender`, and (3) filled with forward-fill (pretend it's time-ordered). Compare their standard deviations.

In [18]:
# your code here


### Exercise 3: Handle Duplicates
Check if the original `df` has any duplicate rows based on `patient_id` (there shouldn't be, but verify). Then create a scenario with 5 duplicate rows and demonstrate how to remove them while keeping the first occurrence.

In [19]:
# your code here


### Exercise 4: Clean String Columns
The `smoking_status` column contains values like 'never', 'former', 'current', and nulls. Standardize all values to lowercase, strip whitespace, and replace nulls with 'unknown'. Count the final distribution.

In [20]:
# your code here


### Exercise 5: Fix Data Types
Convert the `last_visit_date` column to datetime format (if not already). Then extract the month and year into separate columns. Finally, calculate the number of days between the visit date and today.

In [21]:
# your code here


### Exercise 6: Build a Complete Cleaning Function
Write a function `quick_clean(df)` that: (1) drops rows with missing `age` or `heart_disease`, (2) fills `blood_pressure` with the median, (3) fills `bmi` with the mean, (4) converts `gender` to lowercase, and (5) returns the cleaned DataFrame with a print statement showing before/after row count.

In [22]:
# your code here


### Exercise 7: Compare Model-Ready Data
Prepare two datasets: `X_raw` (drop all rows with any missing values) and `X_clean` (apply mean imputation to numerical columns, mode to categorical). Compare their shapes and discuss which would be better for a heart disease prediction model.

In [23]:
# your code here


### Exercise 8: Outlier Flagging
Using the IQR method, identify outliers in the `blood_pressure` column. Create a new column `bp_outlier` that is True for outliers. How many outliers did you find? What are their `patient_id` values?

In [24]:
# your code here


### Exercise 9: Group-Wise Imputation Challenge
Fill missing `blood_pressure` values using the mean within each `gender` group. Then verify that no missing values remain and calculate the mean blood pressure for each gender after imputation.

In [25]:
# your code here


### Exercise 10: Advanced String Cleaning
Create a new column `email_domain` that extracts the domain part (after @) from a hypothetical email column. Handle cases where email might be null. Then count how many unique domains exist.

In [26]:
# your code here


### Exercise 11: Complete Pipeline Application
Apply the `clean_medical_data` function from the tutorial to a subset of the data (first 500 rows). Then create a visualization comparing the distribution of `bmi` before and after cleaning.

In [27]:
# your code here


## Solutions (check after attempting) {#solutions}

### Solution 1: Detect and Summarize Missing Values

In [28]:
# Solution 1
missing_counts = df.isnull().sum()
missing_pct = (missing_counts / len(df) * 100).round(2)

summary = pd.DataFrame({
    'Missing Count': missing_counts,
    'Missing %': missing_pct
}).sort_values('Missing Count', ascending=False)

print("Missing Values Summary:")
print(summary[missing_counts > 0])
print(f"\nColumn with most missing: {missing_counts.idxmax()} ({missing_counts.max()} values)")

# Expected output: cholesterol and smoking_status likely have most missing due to our MAR/MNAR patterns

Missing Values Summary:
                Missing Count  Missing %
cholesterol               102       10.2
smoking_status             85        8.5
blood_pressure             50        5.0
diabetes                   47        4.7
gender                     42        4.2

Column with most missing: cholesterol (102 values)


C:\Users\786\AppData\Local\Temp\ipykernel_2180\2521961393.py:11: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  print(summary[missing_counts > 0])


### Solution 2: Apply Different Imputation Strategies

In [29]:
# Solution 2
# Strategy 1: Overall mean
chol_mean = df['cholesterol'].fillna(df['cholesterol'].mean())

# Strategy 2: Median by gender
gender_median = df.groupby('gender')['cholesterol'].transform('median')
chol_gender_median = df['cholesterol'].fillna(gender_median)

# Strategy 3: Forward fill (for demo, sort by patient_id first)
chol_ffill = df.sort_values('patient_id')['cholesterol'].ffill()

print("Standard Deviation Comparison:")
print(f"Original (non-null): {df['cholesterol'].dropna().std():.2f}")
print(f"Overall mean fill:   {chol_mean.std():.2f}")
print(f"Gender median fill:  {chol_gender_median.std():.2f}")
print(f"Forward fill:        {chol_ffill.std():.2f}")
print("\nNote: Group-wise imputation typically preserves variance better!")

Standard Deviation Comparison:
Original (non-null): 39.43
Overall mean fill:   37.37
Gender median fill:  37.42
Forward fill:        39.42

Note: Group-wise imputation typically preserves variance better!


### Solution 3: Handle Duplicates

In [30]:
# Solution 3
# Check original duplicates by patient_id
dup_by_id = df.duplicated(subset=['patient_id']).sum()
print(f"Original duplicates by patient_id: {dup_by_id}")

# Create scenario with duplicates
df_with_dups = pd.concat([df.iloc[:10], df.iloc[5:8]], ignore_index=True)
print(f"After adding duplicates: {len(df_with_dups)} rows")
print(f"Duplicate rows found: {df_with_dups.duplicated().sum()}")

# Remove keeping first
df_unique = df_with_dups.drop_duplicates(keep='first')
print(f"After removing duplicates: {len(df_unique)} rows")
print(f"Removed: {len(df_with_dups) - len(df_unique)} rows")

Original duplicates by patient_id: 0
After adding duplicates: 13 rows
Duplicate rows found: 3
After removing duplicates: 10 rows
Removed: 3 rows


### Solution 4: Clean String Columns

In [31]:
# Solution 4
df_str = df.copy()
df_str['smoking_status'] = df_str['smoking_status'].str.lower().str.strip()
df_str['smoking_status'] = df_str['smoking_status'].fillna('unknown')

print("Final smoking_status distribution:")
print(df_str['smoking_status'].value_counts())
print(f"\nTotal: {df_str['smoking_status'].count()}")
print(f"Missing: {df_str['smoking_status'].isnull().sum()}")

Final smoking_status distribution:
smoking_status
never      474
former     265
current    176
unknown     85
Name: count, dtype: int64

Total: 1000
Missing: 0


### Solution 5: Fix Data Types

In [32]:
# Solution 5
df_dates = df.copy()
df_dates['last_visit_date'] = pd.to_datetime(df_dates['last_visit_date'])
df_dates['visit_month'] = df_dates['last_visit_date'].dt.month
df_dates['visit_year'] = df_dates['last_visit_date'].dt.year
df_dates['days_since_visit'] = (pd.Timestamp.now() - df_dates['last_visit_date']).dt.days

print("Date columns created:")
print(df_dates[['patient_id', 'last_visit_date', 'visit_month', 'visit_year', 'days_since_visit']].head())
print(f"\nData types:\n{df_dates[['last_visit_date', 'visit_month', 'visit_year']].dtypes}")

Date columns created:
   patient_id last_visit_date  visit_month  visit_year  days_since_visit
0        1000      2024-07-04            7        2024               633
1        1001      2025-03-12            3        2025               382
2        1002      2023-12-19           12        2023               831
3        1003      2023-10-27           10        2023               884
4        1004      2024-11-12           11        2024               502

Data types:
last_visit_date    datetime64[us]
visit_month                 int32
visit_year                  int32
dtype: object


### Solution 6: Build a Complete Cleaning Function

In [33]:
# Solution 6
def quick_clean(df):
    """Quick cleaning function for medical data."""
    before_rows = len(df)
    
    # Drop rows with missing age or heart_disease
    df_clean = df.dropna(subset=['age', 'heart_disease']).copy()
    
    # Fill blood_pressure with median
    df_clean['blood_pressure'] = df_clean['blood_pressure'].fillna(df_clean['blood_pressure'].median())
    
    # Fill bmi with mean
    df_clean['bmi'] = df_clean['bmi'].fillna(df_clean['bmi'].mean())
    
    # Convert gender to lowercase
    df_clean['gender'] = df_clean['gender'].str.lower()
    
    after_rows = len(df_clean)
    print(f"Quick clean complete: {before_rows} -> {after_rows} rows ({before_rows - after_rows} removed)")
    
    return df_clean

# Test it
df_quick = quick_clean(df)
print(f"\nMissing values after quick clean: {df_quick[['age', 'heart_disease', 'blood_pressure', 'bmi']].isnull().sum().sum()}")
print(f"Gender sample: {df_quick['gender'].unique()}")

Quick clean complete: 1000 -> 1000 rows (0 removed)

Missing values after quick clean: 0
Gender sample: <ArrowStringArray>
['f', 'm', nan]
Length: 3, dtype: str


### Solution 7: Compare Model-Ready Data

In [34]:
# Solution 7
# X_raw: drop all rows with any missing
cols_for_model = ['age', 'blood_pressure', 'cholesterol', 'bmi', 'heart_disease']
X_raw = df[cols_for_model].dropna()
y_raw = X_raw['heart_disease']
X_raw = X_raw.drop('heart_disease', axis=1)

# X_clean: impute numerical, drop target missing
df_imputed = df.copy()
for col in ['blood_pressure', 'cholesterol', 'bmi']:
    df_imputed[col] = df_imputed[col].fillna(df_imputed[col].mean())
df_imputed['gender'] = df_imputed['gender'].fillna(df_imputed['gender'].mode()[0])

X_clean = df_imputed[['age', 'blood_pressure', 'cholesterol', 'bmi', 'heart_disease']].dropna()
y_clean = X_clean['heart_disease']
X_clean = X_clean.drop('heart_disease', axis=1)

print("Comparison:")
print(f"X_raw shape (listwise deletion):  {X_raw.shape}")
print(f"X_clean shape (mean imputation):    {X_clean.shape}")
print(f"Samples retained with imputation:   {len(X_clean) - len(X_raw)} more")
print("\nDiscussion: Imputation retains more data but may introduce bias.")
print("For this dataset, imputation is likely better due to MAR pattern (missing related to age).")

Comparison:
X_raw shape (listwise deletion):  (853, 4)
X_clean shape (mean imputation):    (1000, 4)
Samples retained with imputation:   147 more

Discussion: Imputation retains more data but may introduce bias.
For this dataset, imputation is likely better due to MAR pattern (missing related to age).


### Solution 8: Outlier Flagging

In [35]:
# Solution 8
Q1 = df['blood_pressure'].quantile(0.25)
Q3 = df['blood_pressure'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

df['bp_outlier'] = (df['blood_pressure'] < lower) | (df['blood_pressure'] > upper)
outlier_count = df['bp_outlier'].sum()
outlier_ids = df[df['bp_outlier']]['patient_id'].tolist()

print(f"Blood pressure outliers found: {outlier_count}")
print(f"Valid range: {lower:.1f} to {upper:.1f}")
print(f"Outlier patient IDs (first 10): {outlier_ids[:10]}")
print(f"\nOutlier blood pressure values:")
print(df[df['bp_outlier']]['blood_pressure'].describe())

Blood pressure outliers found: 5
Valid range: 66.9 to 173.6
Outlier patient IDs (first 10): [1031, 1086, 1756, 1796, 1859]

Outlier blood pressure values:
count      5.000000
mean      84.480000
std       59.204493
min       52.300000
25%       53.000000
50%       61.500000
75%       65.700000
max      189.900000
Name: blood_pressure, dtype: float64


### Solution 9: Group-Wise Imputation Challenge

In [36]:
# Solution 9
df_group = df.copy()

# Calculate mean by gender
gender_means = df_group.groupby('gender')['blood_pressure'].transform('mean')

# Fill missing with group mean
df_group['blood_pressure'] = df_group['blood_pressure'].fillna(gender_means)

# Verify
remaining_missing = df_group['blood_pressure'].isnull().sum()
print(f"Remaining missing blood_pressure values: {remaining_missing}")

# Calculate means after imputation
print("\nMean blood pressure by gender (after imputation):")
print(df_group.groupby('gender')['blood_pressure'].mean())

# Overall stats
print(f"\nOverall mean: {df_group['blood_pressure'].mean():.1f}")
print(f"Standard deviation: {df_group['blood_pressure'].std():.1f}")

Remaining missing blood_pressure values: 2

Mean blood pressure by gender (after imputation):
gender
F    120.659348
M    120.148000
Name: blood_pressure, dtype: float64

Overall mean: 120.4
Standard deviation: 19.1


### Solution 10: Advanced String Cleaning

In [37]:
# Solution 10
# Create hypothetical email data
np.random.seed(42)
domains = ['gmail.com', 'yahoo.com', 'hospital.org', 'health.net', None]
df['email'] = [f"patient_{i}@{np.random.choice(domains)}" if np.random.random() > 0.1 else None 
               for i in range(len(df))]

# Extract domain
df['email_domain'] = df['email'].str.split('@').str[1]

# Count unique domains (excluding null)
unique_domains = df['email_domain'].nunique()
domain_counts = df['email_domain'].value_counts()

print(f"Unique email domains: {unique_domains}")
print("\nDomain distribution:")
print(domain_counts)
print(f"\nNull emails: {df['email'].isnull().sum()}")
print(f"Null domains after extraction: {df['email_domain'].isnull().sum()}")

Unique email domains: 5

Domain distribution:
email_domain
None            191
gmail.com       188
yahoo.com       182
hospital.org    170
health.net      166
Name: count, dtype: int64

Null emails: 103
Null domains after extraction: 103


### Solution 11: Complete Pipeline Application

In [38]:
# Solution 11
# Apply to subset
df_subset = df.head(500).copy()
df_cleaned_subset, report = clean_medical_data(df_subset)

print("Cleaning Report for Subset:")
print(f"Original: {report['original_shape']}")
print(f"Final: {report['final_shape']}")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Before
axes[0].hist(df_subset['bmi'].dropna(), bins=30, color='skyblue', edgecolor='black', alpha=0.7)
axes[0].set_title('BMI Distribution - Before Cleaning', fontweight='bold')
axes[0].set_xlabel('BMI')
axes[0].set_ylabel('Count')
axes[0].axvline(df_subset['bmi'].mean(), color='red', linestyle='--', label=f'Mean: {df_subset["bmi"].mean():.1f}')
axes[0].legend()

# After
axes[1].hist(df_cleaned_subset['bmi'], bins=30, color='lightgreen', edgecolor='black', alpha=0.7)
axes[1].set_title('BMI Distribution - After Cleaning', fontweight='bold')
axes[1].set_xlabel('BMI')
axes[1].set_ylabel('Count')
axes[1].axvline(df_cleaned_subset['bmi'].mean(), color='red', linestyle='--', label=f'Mean: {df_cleaned_subset["bmi"].mean():.1f}')
axes[1].legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'exercise_pipeline_bmi.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved BMI comparison from pipeline application")

Cleaning Report for Subset:
Original: (500, 14)
Final: (500, 17)
Saved BMI comparison from pipeline application


C:\Users\786\AppData\Local\Temp\ipykernel_2180\3885032212.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
